# Train the emergency-vehicle model on NEW datasets (free Colab GPU)

Downloads several public emergency-vehicle datasets from Roboflow Universe, merges them into one clean 3-class set (ambulance, fire truck, police car), trains YOLOv8 and scores it on images it never saw.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Make a free Roboflow account and copy your **API key** (app.roboflow.com → Settings → API Keys). You paste it in step 3.

Total time: about 30 to 60 minutes. Colab sessions are temporary, so run the last cell to download the results.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: use Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# The cleaned dataset is on main.
BRANCH = 'main'
!git clone --branch $BRANCH https://github.com/aditya182005/Real_Time_AI_Traffic_Monitoring_Violation_Enforcement.git traffic-agent
%cd traffic-agent
!pip -q install roboflow ultralytics "supervision>=0.20,<0.31"

In [ ]:
from getpass import getpass
ROBOFLOW_KEY = getpass('Paste your Roboflow API key: ')

In [ ]:
# (workspace, project, version or None for the newest, force-one-class or None)
# All are public, CC BY 4.0 (check each page before publishing your work).
SOURCES = [
    ('idk-zjlje', 'emergency-vehicles-v59qx', 10, None),
    ('traffic-rbwic', 'emergency-vehicles-snzgj', None, None),
    ('yolo-fn1iu', 'emergency-vehicles-detection-xockh-af7sr', None, None),
    ('vignesh-jqvmq', 'emergency-vehicle-detection-q4kwr', 3, None),
    ('yolo-emergency-recognition', 'ambulance-detection-wdbvs', 1, None),
    ('yolo-emergency-recognition', 'fire-truck-xumw3-jlvpe', 1, 'fire_truck'),
]

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_KEY)
folders = []

for workspace, project_name, version, forced in SOURCES:
    project = rf.workspace(workspace).project(project_name)
    last_error = None

    for number in ([version] if version else range(12, 0, -1)):
        try:
            data = project.version(number).download('yolov8', location=f'downloads/{project_name}')
            break
        except Exception as exc:
            last_error = exc
    else:
        print('SKIPPED', project_name, '->', last_error)
        continue

    folders.append(data.location + (f':{forced}' if forced else ''))
    print('downloaded', project_name, 'v', number)

print(len(folders), 'datasets ready')

In [ ]:
# Merge. READ THE OUTPUT: every class of every dataset is shown with the
# class it was mapped to. An UNKNOWN class is ignored; if it really is an
# ambulance / fire truck / police car, tell me and I will map it.
assert folders, 'No dataset downloaded: check the SKIPPED lines in the previous cell.'
sources = ' '.join(f'--source "{f}"' for f in folders)
# The new datasets contain almost no pictures of ordinary vehicles, so a model trained
# only on them flags every car and van (97% false alarms in our check). The repository's
# own dataset has ~300 checked negatives; adding it fixes that.
INCLUDE_EXISTING = True
extra = '--include-existing data/emergency_vehicle' if INCLUDE_EXISTING else ''
!python scripts/build_emergency_dataset.py $sources $extra

**Check the table above.** Each class should have a few hundred boxes in `train` and 30 or more in `test`, and the **neg** column (pictures with no emergency vehicle) should be roughly 10% of the images in every split. If neg is near zero the model will flag every ordinary vehicle.

In [ ]:
# Look at real labelled images before training (wrong boxes ruin a model).
import cv2, glob, random, matplotlib.pyplot as plt
NAMES = ['ambulance', 'fire_truck', 'police_car']
files = sorted(glob.glob('data/emergency_vehicle_new/images/train/*'))
random.seed(1)
if len(files) < 50:
    raise SystemExit(f'Only {len(files)} training images were built. Scroll up to the merge cell: '
                     'read its output (a "Cannot build" line, SKIPPED downloads, or a large "dropped (unsure)" count) '
                     'and send it to me before training.')
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

for axis, path in zip(axes.flat, random.sample(files, 12)):
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    label = path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'

    for line in open(label).read().split('\n'):
        if line.strip():
            c, x, y, w, h = line.split(); x, y, w, h = (float(v) for v in (x, y, w, h))
            p1 = (int((x - w / 2) * width), int((y - h / 2) * height))
            p2 = (int((x + w / 2) * width), int((y + h / 2) * height))
            cv2.rectangle(image, p1, p2, (255, 0, 0), 3)
            cv2.putText(image, NAMES[int(c)], (p1[0], max(20, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)

    axis.imshow(image); axis.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
# Train. 80 epochs with early stopping; --batch -1 lets Ultralytics choose.
!python scripts/train_emergency.py --data data/emergency_vehicle_new/data.yaml --epochs 80 --device 0 --batch -1 --report emergency_scores.json

The last lines show **mAP@0.5, precision and recall** on the validation images and on the **test** images (never used in training), with each class listed. Quote the **test** numbers in your report. Above 0.8 is good, 0.5 to 0.8 is usable with misses and false alarms, below 0.5 needs better data.

Most dataset photos are close-ups, so real CCTV footage will score lower. Check it on your own video.

In [ ]:
# Try it on a few test images.
from ultralytics import YOLO
import glob
model = YOLO('models/emergency_vehicle.pt')
for path in sorted(glob.glob('data/emergency_vehicle_new/images/test/*'))[:6]:
    model(path)[0].show()

In [ ]:
from google.colab import files
!zip -q results.zip models/emergency_vehicle.pt emergency_scores.json data/emergency_vehicle_new/dataset_report.json runs/detect/emergency_vehicle/results.png runs/detect/emergency_vehicle/confusion_matrix.png
files.download('models/emergency_vehicle.pt')
files.download('results.zip')

## Then, on your own computer
1. Put `emergency_vehicle.pt` at `models/emergency_vehicle.pt` in the project.
2. Run `python check_setup.py --full`.
3. In the app, the **Emergency vehicles** option on *Analyze footage* and *Live camera* is now available.

The weights are git-ignored on purpose. For a server, mount the `models` folder (see `docker-compose.yml`).